# Tutorial F: Building Neural Network Components from Scratch

**A practical introduction to implementing neural network architecture using OOP principles**

---

## References and Further Resources

### Key References
- Goodfellow, I., Bengio, Y., & Courville, A. (2016). *Deep Learning*. MIT Press. Chapter 6: Deep Feedforward Networks.
- Nielsen, M. (2015). *Neural Networks and Deep Learning*. Determination Press. [Online book](http://neuralnetworksanddeeplearning.com/)
- Chollet, F. (2017). *Deep Learning with Python*. Manning Publications. Chapter 2: The mathematical building blocks of neural networks.

### Further Exploration
- PyTorch source code: [github.com/pytorch/pytorch](https://github.com/pytorch/pytorch)
- Karpathy, A. "Micrograd" - A minimal neural network library: [github.com/karpathy/micrograd](https://github.com/karpathy/micrograd)
- 3Blue1Brown: "Neural Networks" video series - excellent visual explanations

---

## Table of Contents

1. [Introduction: From Theory to Implementation](#introduction)
2. [Foundation: Activation Functions](#activation-functions)
3. [Building Blocks: Layer Classes](#layer-classes)
4. [Measuring Error: Loss Functions](#loss-functions)
5. [Learning: Optimizer Classes](#optimizer-classes)
6. [Composition: The Network Class](#network-class)
7. [Training: Putting It All Together](#training)
8. [Real Application: Nonlinear Regression](#real-application)
9. [Architecture Patterns and Design Principles](#design-principles)
10. [Summary and Extensions](#summary)

---

## 1. Introduction: From Theory to Implementation <a id='introduction'></a>

Neural networks are composed of layers that transform input data through learned parameters. Let's build a complete neural network framework from scratch to understand:

- How data flows forward through layers (forward pass)
- How gradients flow backward for learning (backward pass)
- How different components interact through clean interfaces
- Why OOP makes complex systems manageable

### The Challenge We're Solving

Consider approximating a complex function like:
$$f(x) = \sin(2\pi x) + 0.3\cos(4\pi x)$$

Linear regression can't capture this nonlinearity. We need a neural network that can learn arbitrary smooth functions through composition of simple operations.

### Our Architecture

We'll build:
1. **Activation Functions**: Nonlinear transformations
2. **Layers**: Units that process data
3. **Loss Functions**: Measures of prediction error
4. **Optimizers**: Parameter update rules
5. **Network**: Orchestrates everything

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Tuple, Optional, Callable
from abc import ABC, abstractmethod

# Set random seed for reproducibility
np.random.seed(42)

# Plotting configuration
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 4)

---

## 2. Foundation: Activation Functions <a id='activation-functions'></a>

Activation functions introduce nonlinearity, allowing networks to learn complex patterns. Let's implement the most common ones with both forward and backward (derivative) computations.

In [ ]:
class Activation(ABC):
    """Abstract base class for activation functions."""
    
    @abstractmethod
    def forward(self, inputs: np.ndarray) -> np.ndarray:
        """Compute activation output."""
        pass
    
    @abstractmethod
    def backward(self, inputs: np.ndarray, gradient: np.ndarray) -> np.ndarray:
        """Compute gradient with respect to inputs."""
        pass


class ReLU(Activation):
    """Rectified Linear Unit: f(x) = max(0, x).
    
    The most popular activation in deep learning:
    - Simple: linear for x > 0
    - Non-saturating: no vanishing gradient for positive values
    - Efficient: just a threshold operation
    """
    
    def forward(self, inputs: np.ndarray) -> np.ndarray:
        return np.maximum(0, inputs)
    
    def backward(self, inputs: np.ndarray, gradient: np.ndarray) -> np.ndarray:
        # Derivative is 1 where input > 0, else 0
        return gradient * (inputs > 0)


class Tanh(Activation):
    """Hyperbolic tangent: f(x) = tanh(x).
    
    Output range [-1, 1], zero-centered:
    - Smooth nonlinearity
    - Zero-centered outputs help with gradient flow
    - Can saturate for large |x|
    """
    
    def forward(self, inputs: np.ndarray) -> np.ndarray:
        return np.tanh(inputs)
    
    def backward(self, inputs: np.ndarray, gradient: np.ndarray) -> np.ndarray:
        # Derivative: 1 - tanh^2(x)
        tanh_output = np.tanh(inputs)
        return gradient * (1 - tanh_output**2)


class Sigmoid(Activation):
    """Sigmoid: f(x) = 1 / (1 + exp(-x)).
    
    Output range [0, 1], useful for probabilities:
    - Smooth, differentiable
    - Interpretable as probability
    - Can saturate, causing vanishing gradients
    """
    
    def forward(self, inputs: np.ndarray) -> np.ndarray:
        # Numerically stable implementation
        return 1 / (1 + np.exp(-np.clip(inputs, -500, 500)))
    
    def backward(self, inputs: np.ndarray, gradient: np.ndarray) -> np.ndarray:
        # Derivative: sigmoid(x) * (1 - sigmoid(x))
        sigmoid_output = self.forward(inputs)
        return gradient * sigmoid_output * (1 - sigmoid_output)


class Linear(Activation):
    """Identity activation: f(x) = x.
    
    Useful for:
    - Output layers in regression
    - Debugging (removes nonlinearity)
    """
    
    def forward(self, inputs: np.ndarray) -> np.ndarray:
        return inputs
    
    def backward(self, inputs: np.ndarray, gradient: np.ndarray) -> np.ndarray:
        return gradient

### Visualizing Activation Functions

Let's see how these functions transform inputs and how their derivatives behave:

In [ ]:
def visualize_activations() -> None:
    """Plot activation functions and their derivatives."""
    
    inputs = np.linspace(-3, 3, 200)
    activations = {
        'ReLU': ReLU(),
        'Tanh': Tanh(),
        'Sigmoid': Sigmoid()
    }
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    for idx, (name, activation) in enumerate(activations.items()):
        # Forward pass
        outputs = activation.forward(inputs)
        
        # Backward pass (derivative)
        derivatives = activation.backward(inputs, np.ones_like(inputs))
        
        # Plot
        axes[idx].plot(inputs, outputs, label='Output', linewidth=2)
        axes[idx].plot(inputs, derivatives, label='Derivative', 
                      linewidth=2, linestyle='--', alpha=0.7)
        axes[idx].axhline(y=0, color='k', linestyle='-', alpha=0.3)
        axes[idx].axvline(x=0, color='k', linestyle='-', alpha=0.3)
        axes[idx].set_title(f'{name} Activation')
        axes[idx].set_xlabel('Input')
        axes[idx].set_ylabel('Value')
        axes[idx].legend()
        axes[idx].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

visualize_activations()

**Key observations:**
- **ReLU**: Derivative is 0 for negative inputs ("dead neurons" problem)
- **Tanh**: Derivative vanishes for large |x| (saturation)
- **Sigmoid**: Similar saturation, but stronger (derivatives near 0 at extremes)

---

## 3. Building Blocks: Layer Classes <a id='layer-classes'></a>

Layers are the computational units of neural networks. Each layer:
1. Transforms inputs via learned parameters
2. Applies an activation function
3. Computes gradients during backpropagation

In [ ]:
class Layer:
    """Fully connected (dense) layer with activation.
    
    Computes: activation(X @ W + b)
    where:
        X: input matrix (batch_size, input_dim)
        W: weight matrix (input_dim, output_dim)
        b: bias vector (output_dim,)
    """
    
    def __init__(self, 
                 input_dim: int, 
                 output_dim: int, 
                 activation: Activation,
                 weight_init: str = 'xavier') -> None:
        """
        Initialize layer with random weights.
        
        Args:
            input_dim: Number of input features
            output_dim: Number of output neurons
            activation: Activation function instance
            weight_init: Initialization scheme ('xavier' or 'he')
        """
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.activation = activation
        
        # Initialize weights using appropriate scheme
        if weight_init == 'xavier':
            # Xavier/Glorot initialization: good for tanh, sigmoid
            limit = np.sqrt(6 / (input_dim + output_dim))
            self.weights = np.random.uniform(-limit, limit, 
                                            (input_dim, output_dim))
        elif weight_init == 'he':
            # He initialization: good for ReLU
            std = np.sqrt(2 / input_dim)
            self.weights = np.random.normal(0, std, 
                                           (input_dim, output_dim))
        else:
            raise ValueError(f"Unknown initialization: {weight_init}")
        
        # Initialize biases to zero
        self.biases = np.zeros(output_dim)
        
        # Cache for backward pass
        self.cache = {}
    
    def forward(self, inputs: np.ndarray) -> np.ndarray:
        """
        Forward pass through the layer.
        
        Args:
            inputs: Input data (batch_size, input_dim)
            
        Returns:
            Activated outputs (batch_size, output_dim)
        """
        # Linear transformation
        linear_output = inputs @ self.weights + self.biases
        
        # Apply activation
        activated_output = self.activation.forward(linear_output)
        
        # Cache values needed for backward pass
        self.cache['inputs'] = inputs
        self.cache['linear_output'] = linear_output
        
        return activated_output
    
    def backward(self, output_gradient: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        """
        Backward pass: compute gradients.
        
        Args:
            output_gradient: Gradient from next layer (batch_size, output_dim)
            
        Returns:
            Tuple of (input_gradient, weight_gradient, bias_gradient)
        """
        inputs = self.cache['inputs']
        linear_output = self.cache['linear_output']
        batch_size = inputs.shape[0]
        
        # Gradient through activation
        activation_gradient = self.activation.backward(linear_output, output_gradient)
        
        # Gradient with respect to weights: X^T @ dL/dZ
        weight_gradient = inputs.T @ activation_gradient / batch_size
        
        # Gradient with respect to biases: sum over batch
        bias_gradient = np.mean(activation_gradient, axis=0)
        
        # Gradient with respect to inputs: dL/dZ @ W^T
        input_gradient = activation_gradient @ self.weights.T
        
        return input_gradient, weight_gradient, bias_gradient
    
    def __repr__(self) -> str:
        return (f"Layer(input_dim={self.input_dim}, "
                f"output_dim={self.output_dim}, "
                f"activation={self.activation.__class__.__name__})")

### Understanding Weight Initialization

Weight initialization is crucial for training:
- **Too small**: Vanishing gradients (signal dies)
- **Too large**: Exploding gradients (training unstable)
- **Xavier**: Preserves variance for symmetric activations (tanh, sigmoid)
- **He**: Accounts for ReLU's zero gradient for negative values

Let's verify our initialization preserves signal variance:

In [ ]:
def test_weight_initialization() -> None:
    """Verify that weight initialization preserves variance."""
    
    input_dim, output_dim = 100, 100
    num_samples = 1000
    
    # Generate random input
    inputs = np.random.randn(num_samples, input_dim)
    
    # Test different initializations
    init_schemes = ['xavier', 'he']
    activations = [Tanh(), ReLU()]
    
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    
    for idx, (init, act) in enumerate(zip(init_schemes, activations)):
        layer = Layer(input_dim, output_dim, act, weight_init=init)
        
        # Track variance through multiple layers
        variances = [np.var(inputs)]
        current = inputs
        
        for _ in range(10):
            current = layer.forward(current)
            variances.append(np.var(current))
        
        # Plot variance evolution
        row, col = idx // 2, idx % 2
        axes[row, col].plot(variances, marker='o', linewidth=2)
        axes[row, col].axhline(y=1, color='r', linestyle='--', 
                              label='Target variance = 1', alpha=0.7)
        axes[row, col].set_title(f'{init.capitalize()} init + {act.__class__.__name__}')
        axes[row, col].set_xlabel('Layer depth')
        axes[row, col].set_ylabel('Variance')
        axes[row, col].legend()
        axes[row, col].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

test_weight_initialization()

**Key insight**: He initialization maintains variance better with ReLU because it accounts for the fact that ReLU zeros out half the values.

---

## 4. Measuring Error: Loss Functions <a id='loss-functions'></a>

Loss functions quantify how far predictions are from targets. They must be differentiable for gradient-based optimization.

In [ ]:
class Loss(ABC):
    """Abstract base class for loss functions."""
    
    @abstractmethod
    def compute(self, predictions: np.ndarray, targets: np.ndarray) -> float:
        """Calculate loss value."""
        pass
    
    @abstractmethod
    def gradient(self, predictions: np.ndarray, targets: np.ndarray) -> np.ndarray:
        """Calculate gradient of loss with respect to predictions."""
        pass


class MeanSquaredError(Loss):
    """Mean Squared Error loss for regression.
    
    L = (1/n) * sum((y_pred - y_true)^2)
    
    Properties:
    - Penalizes large errors more heavily (quadratic)
    - Differentiable everywhere
    - Sensitive to outliers
    """
    
    def compute(self, predictions: np.ndarray, targets: np.ndarray) -> float:
        return np.mean((predictions - targets)**2)
    
    def gradient(self, predictions: np.ndarray, targets: np.ndarray) -> np.ndarray:
        # Derivative: 2 * (y_pred - y_true) / n
        return 2 * (predictions - targets) / predictions.shape[0]


class MeanAbsoluteError(Loss):
    """Mean Absolute Error loss for regression.
    
    L = (1/n) * sum(|y_pred - y_true|)
    
    Properties:
    - Linear penalty (more robust to outliers)
    - Not differentiable at zero (handled with sign function)
    - Treats all errors equally
    """
    
    def compute(self, predictions: np.ndarray, targets: np.ndarray) -> float:
        return np.mean(np.abs(predictions - targets))
    
    def gradient(self, predictions: np.ndarray, targets: np.ndarray) -> np.ndarray:
        # Derivative: sign(y_pred - y_true) / n
        return np.sign(predictions - targets) / predictions.shape[0]


class HuberLoss(Loss):
    """Huber loss: quadratic near zero, linear for large errors.
    
    Combines benefits of MSE and MAE:
    - Smooth and differentiable everywhere
    - Less sensitive to outliers than MSE
    - More stable gradients than MAE near zero
    """
    
    def __init__(self, delta: float = 1.0) -> None:
        """
        Args:
            delta: Threshold for switching between quadratic and linear
        """
        self.delta = delta
    
    def compute(self, predictions: np.ndarray, targets: np.ndarray) -> float:
        errors = predictions - targets
        abs_errors = np.abs(errors)
        
        # Quadratic for small errors, linear for large
        quadratic = 0.5 * errors**2
        linear = self.delta * abs_errors - 0.5 * self.delta**2
        
        return np.mean(np.where(abs_errors <= self.delta, quadratic, linear))
    
    def gradient(self, predictions: np.ndarray, targets: np.ndarray) -> np.ndarray:
        errors = predictions - targets
        abs_errors = np.abs(errors)
        
        # Gradient is error for small, delta*sign(error) for large
        gradient = np.where(abs_errors <= self.delta,
                          errors,
                          self.delta * np.sign(errors))
        
        return gradient / predictions.shape[0]

### Comparing Loss Functions

Let's visualize how different losses handle errors:

In [ ]:
def compare_losses() -> None:
    """Compare behavior of different loss functions."""
    
    errors = np.linspace(-3, 3, 200)
    predictions = errors.reshape(-1, 1)
    targets = np.zeros_like(predictions)
    
    losses = {
        'MSE': MeanSquaredError(),
        'MAE': MeanAbsoluteError(),
        'Huber': HuberLoss(delta=1.0)
    }
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    for name, loss in losses.items():
        # Compute loss values
        loss_values = [loss.compute(pred.reshape(-1, 1), targets[:1]) 
                      for pred in predictions]
        
        # Compute gradients
        gradients = [loss.gradient(pred.reshape(-1, 1), targets[:1])[0, 0] 
                    for pred in predictions]
        
        ax1.plot(errors, loss_values, label=name, linewidth=2)
        ax2.plot(errors, gradients, label=name, linewidth=2)
    
    ax1.set_title('Loss Value vs. Error')
    ax1.set_xlabel('Prediction Error')
    ax1.set_ylabel('Loss')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    ax2.set_title('Loss Gradient vs. Error')
    ax2.set_xlabel('Prediction Error')
    ax2.set_ylabel('Gradient')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

compare_losses()

**Observations:**
- **MSE**: Gradients grow linearly with error (can explode for outliers)
- **MAE**: Constant gradient magnitude (more stable but can be slow)
- **Huber**: Best of both worlds (smooth near zero, stable for large errors)

---

## 5. Learning: Optimizer Classes <a id='optimizer-classes'></a>

Optimizers update parameters using gradients. Different optimizers have different convergence properties.

In [ ]:
class Optimizer(ABC):
    """Abstract base class for optimizers."""
    
    @abstractmethod
    def update(self, params: np.ndarray, gradients: np.ndarray, param_name: str) -> np.ndarray:
        """Update parameters using gradients."""
        pass


class SGD(Optimizer):
    """Stochastic Gradient Descent with momentum.
    
    Update rule:
    v_t = momentum * v_{t-1} + gradient
    param_t = param_{t-1} - learning_rate * v_t
    
    Momentum helps:
    - Accelerate in consistent directions
    - Dampen oscillations
    - Escape local minima
    """
    
    def __init__(self, learning_rate: float = 0.01, momentum: float = 0.9) -> None:
        """
        Args:
            learning_rate: Step size for updates
            momentum: Exponential decay factor for velocity
        """
        self.learning_rate = learning_rate
        self.momentum = momentum
        self.velocity = {}  # Store velocity for each parameter
    
    def update(self, params: np.ndarray, gradients: np.ndarray, param_name: str) -> np.ndarray:
        # Initialize velocity if first time
        if param_name not in self.velocity:
            self.velocity[param_name] = np.zeros_like(params)
        
        # Update velocity with momentum
        self.velocity[param_name] = (self.momentum * self.velocity[param_name] + 
                                     gradients)
        
        # Update parameters
        return params - self.learning_rate * self.velocity[param_name]


class Adam(Optimizer):
    """Adam: Adaptive Moment Estimation.
    
    Maintains adaptive learning rates for each parameter:
    - First moment (mean) of gradients
    - Second moment (variance) of gradients
    - Bias correction for initialization
    
    Generally the best default optimizer:
    - Fast convergence
    - Robust to hyperparameters
    - Works well across many problems
    """
    
    def __init__(self, 
                 learning_rate: float = 0.001,
                 beta1: float = 0.9,
                 beta2: float = 0.999,
                 epsilon: float = 1e-8) -> None:
        """
        Args:
            learning_rate: Step size
            beta1: Exponential decay for first moment
            beta2: Exponential decay for second moment
            epsilon: Small constant for numerical stability
        """
        self.learning_rate = learning_rate
        self.beta1 = beta1
        self.beta2 = beta2
        self.epsilon = epsilon
        
        # State for each parameter
        self.mean = {}      # First moment
        self.variance = {}  # Second moment
        self.timestep = {}  # For bias correction
    
    def update(self, params: np.ndarray, gradients: np.ndarray, param_name: str) -> np.ndarray:
        # Initialize state if first time
        if param_name not in self.mean:
            self.mean[param_name] = np.zeros_like(params)
            self.variance[param_name] = np.zeros_like(params)
            self.timestep[param_name] = 0
        
        self.timestep[param_name] += 1
        current_timestep = self.timestep[param_name]
        
        # Update biased first moment estimate
        self.mean[param_name] = (self.beta1 * self.mean[param_name] + 
                                 (1 - self.beta1) * gradients)
        
        # Update biased second moment estimate
        self.variance[param_name] = (self.beta2 * self.variance[param_name] + 
                                     (1 - self.beta2) * gradients**2)
        
        # Bias correction
        mean_corrected = self.mean[param_name] / (1 - self.beta1**current_timestep)
        variance_corrected = self.variance[param_name] / (1 - self.beta2**current_timestep)
        
        # Update parameters with adaptive learning rate
        update = self.learning_rate * mean_corrected / (np.sqrt(variance_corrected) + self.epsilon)
        
        return params - update

---

## 6. Composition: The Network Class <a id='network-class'></a>

The Network class orchestrates layers, loss, and optimization into a complete learning system.

In [ ]:
class NeuralNetwork:
    """Feedforward neural network.
    
    Manages:
    - Forward propagation through layers
    - Backward propagation of gradients
    - Parameter updates via optimizer
    - Training history and metrics
    """
    
    def __init__(self, layers: List[Layer], loss: Loss, optimizer: Optimizer) -> None:
        """
        Args:
            layers: List of Layer instances
            loss: Loss function instance
            optimizer: Optimizer instance
        """
        self.layers = layers
        self.loss_fn = loss
        self.optimizer = optimizer
        self.history = {'loss': []}
    
    def forward(self, inputs: np.ndarray) -> np.ndarray:
        """
        Forward pass through all layers.
        
        Args:
            inputs: Input data (batch_size, input_dim)
            
        Returns:
            Network output (batch_size, output_dim)
        """
        current = inputs
        for layer in self.layers:
            current = layer.forward(current)
        return current
    
    def backward(self, loss_gradient: np.ndarray) -> None:
        """
        Backward pass through all layers.
        
        Args:
            loss_gradient: Gradient of loss w.r.t. network output
        """
        current_gradient = loss_gradient
        
        # Backpropagate through layers in reverse order
        for layer in reversed(self.layers):
            input_grad, weight_grad, bias_grad = layer.backward(current_gradient)
            current_gradient = input_grad
            
            # Update parameters using optimizer
            layer.weights = self.optimizer.update(
                layer.weights, weight_grad, f"weights_{id(layer)}"
            )
            layer.biases = self.optimizer.update(
                layer.biases, bias_grad, f"biases_{id(layer)}"
            )
    
    def train_step(self, inputs: np.ndarray, targets: np.ndarray) -> float:
        """
        Single training step.
        
        Args:
            inputs: Training data
            targets: Target values
            
        Returns:
            Loss value for this batch
        """
        # Forward pass
        predictions = self.forward(inputs)
        
        # Compute loss
        loss = self.loss_fn.compute(predictions, targets)
        
        # Backward pass
        loss_gradient = self.loss_fn.gradient(predictions, targets)
        self.backward(loss_gradient)
        
        return loss
    
    def fit(self, 
            inputs: np.ndarray, 
            targets: np.ndarray, 
            epochs: int,
            batch_size: int = 32,
            verbose: bool = True) -> None:
        """
        Train the network.
        
        Args:
            inputs: Training data
            targets: Target values
            epochs: Number of training epochs
            batch_size: Size of mini-batches
            verbose: Whether to print progress
        """
        num_samples = inputs.shape[0]
        
        for epoch in range(epochs):
            # Shuffle data
            indices = np.random.permutation(num_samples)
            inputs_shuffled = inputs[indices]
            targets_shuffled = targets[indices]
            
            # Mini-batch training
            epoch_losses = []
            for start_idx in range(0, num_samples, batch_size):
                end_idx = min(start_idx + batch_size, num_samples)
                batch_inputs = inputs_shuffled[start_idx:end_idx]
                batch_targets = targets_shuffled[start_idx:end_idx]
                
                loss = self.train_step(batch_inputs, batch_targets)
                epoch_losses.append(loss)
            
            # Record average loss for epoch
            avg_loss = np.mean(epoch_losses)
            self.history['loss'].append(avg_loss)
            
            if verbose and (epoch + 1) % max(1, epochs // 10) == 0:
                print(f"Epoch {epoch + 1}/{epochs}, Loss: {avg_loss:.6f}")
    
    def predict(self, inputs: np.ndarray) -> np.ndarray:
        """
        Make predictions on new data.
        
        Args:
            inputs: Input data
            
        Returns:
            Network predictions
        """
        return self.forward(inputs)

---

## 7. Training: Putting It All Together <a id='training'></a>

Let's test our framework on a simple problem first: learning the XOR function, which is not linearly separable.

In [ ]:
def test_xor_problem() -> None:
    """Train network to learn XOR function."""
    
    # XOR dataset
    inputs = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
    targets = np.array([[0], [1], [1], [0]])
    
    # Build network: 2 inputs -> 4 hidden -> 1 output
    network = NeuralNetwork(
        layers=[
            Layer(2, 4, ReLU(), weight_init='he'),
            Layer(4, 1, Sigmoid(), weight_init='xavier')
        ],
        loss=MeanSquaredError(),
        optimizer=Adam(learning_rate=0.1)
    )
    
    print("Training XOR network...")
    network.fit(inputs, targets, epochs=1000, batch_size=4, verbose=False)
    
    # Test predictions
    predictions = network.predict(inputs)
    
    print("\nResults:")
    for inp, target, pred in zip(inputs, targets, predictions):
        print(f"Input: {inp} | Target: {target[0]:.0f} | "
              f"Prediction: {pred[0]:.4f} | "
              f"Correct: {abs(pred[0] - target[0]) < 0.1}")
    
    # Plot training curve
    plt.figure(figsize=(10, 4))
    plt.plot(network.history['loss'], linewidth=2)
    plt.title('XOR Training Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.grid(True, alpha=0.3)
    plt.yscale('log')
    plt.show()

test_xor_problem()

**Success!** The network learns XOR, which requires nonlinearity to solve.

---

## 8. Real Application: Nonlinear Regression <a id='real-application'></a>

Now let's apply our framework to a realistic problem: approximating a complex periodic function.

In [ ]:
def nonlinear_regression_demo() -> None:
    """Demonstrate neural network for function approximation."""
    
    # Generate complex target function
    def target_function(x_values: np.ndarray) -> np.ndarray:
        """Complex periodic function with multiple frequencies."""
        return (np.sin(2 * np.pi * x_values) + 
                0.3 * np.cos(4 * np.pi * x_values) + 
                0.1 * np.sin(8 * np.pi * x_values))
    
    # Generate training data
    num_samples = 200
    x_train = np.random.uniform(0, 1, (num_samples, 1))
    y_train = target_function(x_train) + np.random.normal(0, 0.05, (num_samples, 1))
    
    # Generate test data (denser sampling)
    x_test = np.linspace(0, 1, 500).reshape(-1, 1)
    y_test = target_function(x_test)
    
    # Build network with multiple hidden layers
    network = NeuralNetwork(
        layers=[
            Layer(1, 32, ReLU(), weight_init='he'),
            Layer(32, 32, ReLU(), weight_init='he'),
            Layer(32, 16, ReLU(), weight_init='he'),
            Layer(16, 1, Linear(), weight_init='xavier')
        ],
        loss=HuberLoss(delta=0.5),
        optimizer=Adam(learning_rate=0.01)
    )
    
    print("Training neural network for function approximation...")
    network.fit(x_train, y_train, epochs=500, batch_size=32, verbose=True)
    
    # Make predictions
    y_pred = network.predict(x_test)
    
    # Visualize results
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # Plot function approximation
    ax1.scatter(x_train, y_train, alpha=0.3, s=20, label='Training data')
    ax1.plot(x_test, y_test, 'g-', linewidth=2, label='True function', alpha=0.7)
    ax1.plot(x_test, y_pred, 'r--', linewidth=2, label='Network prediction')
    ax1.set_title('Neural Network Function Approximation')
    ax1.set_xlabel('x')
    ax1.set_ylabel('y')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Plot training history
    ax2.plot(network.history['loss'], linewidth=2)
    ax2.set_title('Training Loss Over Time')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.set_yscale('log')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Calculate error metrics
    mse = np.mean((y_pred - y_test)**2)
    mae = np.mean(np.abs(y_pred - y_test))
    print(f"\nTest Set Performance:")
    print(f"Mean Squared Error: {mse:.6f}")
    print(f"Mean Absolute Error: {mae:.6f}")

nonlinear_regression_demo()

### Exploring Network Capacity

How does network architecture affect performance? Let's compare different configurations:

In [ ]:
def compare_architectures() -> None:
    """Compare different network architectures on the same task."""
    
    # Generate data
    def target_function(x_values: np.ndarray) -> np.ndarray:
        return np.sin(2 * np.pi * x_values) + 0.3 * np.cos(4 * np.pi * x_values)
    
    num_samples = 150
    x_train = np.random.uniform(0, 1, (num_samples, 1))
    y_train = target_function(x_train) + np.random.normal(0, 0.05, (num_samples, 1))
    
    x_test = np.linspace(0, 1, 300).reshape(-1, 1)
    y_test = target_function(x_test)
    
    # Define different architectures
    architectures = {
        'Shallow (1 layer)': [
            Layer(1, 32, ReLU(), weight_init='he'),
            Layer(32, 1, Linear(), weight_init='xavier')
        ],
        'Medium (2 layers)': [
            Layer(1, 16, ReLU(), weight_init='he'),
            Layer(16, 16, ReLU(), weight_init='he'),
            Layer(16, 1, Linear(), weight_init='xavier')
        ],
        'Deep (3 layers)': [
            Layer(1, 16, ReLU(), weight_init='he'),
            Layer(16, 16, ReLU(), weight_init='he'),
            Layer(16, 8, ReLU(), weight_init='he'),
            Layer(8, 1, Linear(), weight_init='xavier')
        ]
    }
    
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    for idx, (name, layers) in enumerate(architectures.items()):
        print(f"\nTraining {name} network...")
        
        network = NeuralNetwork(
            layers=layers,
            loss=MeanSquaredError(),
            optimizer=Adam(learning_rate=0.01)
        )
        
        network.fit(x_train, y_train, epochs=300, batch_size=32, verbose=False)
        
        y_pred = network.predict(x_test)
        mse = np.mean((y_pred - y_test)**2)
        
        # Plot
        axes[idx].scatter(x_train, y_train, alpha=0.3, s=15, label='Data')
        axes[idx].plot(x_test, y_test, 'g-', linewidth=2, label='True', alpha=0.7)
        axes[idx].plot(x_test, y_pred, 'r--', linewidth=2, label='Prediction')
        axes[idx].set_title(f'{name}\nMSE: {mse:.4f}')
        axes[idx].set_xlabel('x')
        axes[idx].set_ylabel('y')
        axes[idx].legend()
        axes[idx].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

compare_architectures()

**Insights:**
- **Shallow**: Fast training, may underfit complex functions
- **Medium**: Good balance for many tasks
- **Deep**: More expressive, but requires more data and tuning

---

## 9. Architecture Patterns and Design Principles <a id='design-principles'></a>

Our implementation demonstrates key software engineering principles for ML systems:

### 1. Separation of Concerns

Each class has a single, well-defined responsibility:
- **Activation**: Nonlinear transformation
- **Layer**: Linear transformation + activation
- **Loss**: Error measurement
- **Optimizer**: Parameter updates
- **Network**: Orchestration

### 2. Interface-Based Design

Abstract base classes define contracts:

In [ ]:
def demonstrate_polymorphism() -> None:
    """Show how interfaces enable flexible component swapping."""
    
    # Same data, different components
    x_data = np.random.randn(100, 2)
    y_data = np.random.randn(100, 1)
    
    # Try different combinations
    configurations = [
        ('ReLU + Adam + MSE', ReLU(), Adam(), MeanSquaredError()),
        ('Tanh + SGD + MAE', Tanh(), SGD(momentum=0.9), MeanAbsoluteError()),
        ('Sigmoid + Adam + Huber', Sigmoid(), Adam(), HuberLoss())
    ]
    
    for name, activation, optimizer, loss in configurations:
        network = NeuralNetwork(
            layers=[
                Layer(2, 8, activation, weight_init='he'),
                Layer(8, 1, Linear(), weight_init='xavier')
            ],
            loss=loss,
            optimizer=optimizer
        )
        
        network.fit(x_data, y_data, epochs=50, batch_size=32, verbose=False)
        final_loss = network.history['loss'][-1]
        print(f"{name}: Final loss = {final_loss:.4f}")

demonstrate_polymorphism()

### 3. Composition Over Inheritance

Networks are composed of layers, not inherited from them:
- More flexible: easily add/remove layers
- Better modularity: layers are independent
- Clearer relationships: has-a vs. is-a

### 4. Encapsulation

Internal state (caches, velocities) is hidden:
- Users interact through clean interfaces
- Implementation details can change
- Reduces coupling between components

### 5. Professional Framework Comparison

Our design mirrors professional frameworks:

```python
# PyTorch
model = nn.Sequential(
    nn.Linear(1, 32),
    nn.ReLU(),
    nn.Linear(32, 1)
)
optimizer = optim.Adam(model.parameters())
criterion = nn.MSELoss()

# Our framework
network = NeuralNetwork(
    layers=[
        Layer(1, 32, ReLU()),
        Layer(32, 1, Linear())
    ],
    loss=MeanSquaredError(),
    optimizer=Adam()
)
```

The patterns are the same because they solve the same problems!

---

## 10. Summary and Extensions <a id='summary'></a>

### What We've Built

A complete neural network framework with:

1. **Forward propagation**: Data flows through layers
2. **Backpropagation**: Gradients flow backward
3. **Optimization**: Parameters learn from gradients
4. **Modular design**: Swappable components
5. **Real applications**: Function approximation

### Key Takeaways

**Mathematical Concepts**:
- Neural networks approximate functions through composition
- Backpropagation uses chain rule to compute gradients
- Different activations have different properties
- Loss functions shape the optimization landscape

**Software Engineering**:
- OOP enables modularity and reusability
- Abstract interfaces define contracts
- Composition provides flexibility
- Encapsulation hides complexity

**Practical Insights**:
- Weight initialization matters for training stability
- Architecture choice depends on problem complexity
- Different optimizers converge differently
- Loss functions affect learning behavior

### Extensions to Explore

How might you extend this framework?

1. **Regularization**:
   - Add L1/L2 penalties to prevent overfitting
   - Implement dropout for better generalization

2. **More Layer Types**:
   - Convolutional layers for image data
   - Recurrent layers for sequences
   - Batch normalization for training stability

3. **Advanced Optimizers**:
   - AdaGrad, RMSprop for adaptive learning rates
   - Learning rate schedules (decay, warmup)

4. **Better Training**:
   - Early stopping to prevent overfitting
   - Validation set monitoring
   - Gradient clipping for stability

5. **New Problem Types**:
   - Classification with softmax and cross-entropy
   - Multi-task learning
   - Autoencoders for unsupervised learning

### From Scratch to Production

This framework teaches fundamentals, but for real work:

- **Use established frameworks**: PyTorch, TensorFlow
- **They provide**:
  - GPU acceleration (orders of magnitude faster)
  - Automatic differentiation (no manual gradients)
  - Optimized operations (highly efficient)
  - Rich ecosystems (pre-trained models, datasets)

But understanding these fundamentals helps you:
- Debug training problems
- Design better architectures
- Implement custom components
- Reason about model behavior

### The Journey

We've gone from:
- Basic Python and data structures
- Object-oriented programming principles
- Mathematical abstractions as classes
- **To building a complete ML system!**

This progression mirrors real ML engineering:
1. Understand the mathematics
2. Design clean abstractions
3. Implement modular components
4. Compose into complete systems
5. Test and iterate

---

## Congratulations!

You've completed this tutorial series on Python fundamentals and machine learning. You now have:

- Strong Python programming skills
- Understanding of OOP principles
- Experience with scientific computing libraries
- A complete neural network implementation
- Foundation for advanced ML topics

The skills you've developed—mathematical thinking, software design, and systematic problem-solving—are transferable to any area of computer science and data science.

Keep building, keep learning, and enjoy the journey!